# Gold Price Prediction (Time Series)

**Goal:** Predict the **daily gold price** (closing price of Gold Futures, USD per troy ounce).

**Data source:** [Yahoo Finance - Gold Futures (GC=F)](https://finance.yahoo.com/quote/GC=F/history/), downloaded with the `yfinance` library (daily data from 2005).

## Notebook plan
1. Load the data
2. Data cleaning (dates, duplicates, missing days, outliers)
3. EDA (trend, moving averages, yearly average, returns, seasonality, stationarity, ACF / PACF)
4. Feature creation (lag features for machine learning)
5. **Data splitting based on time** - 80% train (oldest) / 15% test / 5% unseen (newest)
6. Modeling - Naive, Moving Average, Holt Exponential Smoothing, ARIMA, and 6 machine learning models
7. Training and testing evaluation (MAE, RMSE, MAPE, R2)
8. Compare the models, save the best, load it, predict the last 5%
9. Forecast the next 30 business days

## 0. Import libraries

In [ ]:
# If a library is missing, run this in a cell:
# !pip install pandas numpy matplotlib seaborn statsmodels scikit-learn xgboost joblib

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 123

## 1. Load the data

In [ ]:
df = pd.read_csv("data/gold_price_daily.csv", parse_dates=["Date"])
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 2. Data cleaning

### 2.1 Rename columns, sort by date, remove duplicate dates
In time series the **order of the rows is very important**, so we sort by date and use the date as the index.

In [ ]:
df = df.rename(columns={"Date": "date", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"})
df = df.sort_values("date")

print("Duplicate dates:", df["date"].duplicated().sum())
df = df.drop_duplicates(subset="date")

df = df.set_index("date")
print("From", df.index.min().date(), "to", df.index.max().date())
df.head()

### 2.2 Missing values

In [ ]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().sum() / len(df) * 100).round(2),
})
missing

### 2.3 Missing days
The market is closed on weekends and holidays. We use **business days** (Monday - Friday) as the frequency with `asfreq("B")`.
Holidays become empty rows, and we fill them with the **previous day's price** (`ffill`) - the price did not change because there was no trading.

We only need the `close` price to predict.

In [ ]:
data = df[["close"]].asfreq("B")
print("Empty business days (holidays):", data["close"].isna().sum())

data["close"] = data["close"].ffill()
print("Missing after ffill:", data["close"].isna().sum())
print("Rows:", len(data))
data.head()

### 2.4 Outliers
In price data we **do not remove outliers** - a big jump is a real market event. We check for impossible values (price <= 0) and look at the daily % changes.

In [ ]:
print("Prices <= 0:", (data["close"] <= 0).sum())

daily_return = data["close"].pct_change() * 100
print(daily_return.describe())

plt.figure(figsize=(10, 3))
sns.boxplot(x=daily_return.dropna())
plt.title("Daily % change of the gold price")
plt.show()

## 3. EDA

### 3.1 Gold price over time

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(data.index, data["close"])
plt.title("Gold price (USD / ounce)")
plt.ylabel("USD")
plt.show()

### 3.2 Moving averages (30 days and 200 days)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(data["close"], label="Close", alpha=0.5)
plt.plot(data["close"].rolling(30).mean(), label="30-day average")
plt.plot(data["close"].rolling(200).mean(), label="200-day average")
plt.title("Gold price with moving averages")
plt.legend()
plt.show()

### 3.3 Average price per year

In [ ]:
yearly = data["close"].resample("YE").mean()
yearly.index = yearly.index.year
yearly.plot(kind="bar", figsize=(14, 4), color="goldenrod")
plt.title("Average gold price per year")
plt.ylabel("USD")
plt.show()

### 3.4 Daily returns (% change) distribution

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(daily_return.dropna(), bins=100, kde=True)
plt.title("Daily % change")
plt.show()

### 3.5 Seasonal decomposition (monthly data)
We split the monthly average price into **trend**, **seasonality** (repeating every 12 months) and **residual** (noise).

In [ ]:
monthly = data["close"].resample("ME").mean()
decomposition = seasonal_decompose(monthly, model="multiplicative", period=12)
fig = decomposition.plot()
fig.set_size_inches(14, 8)
plt.show()

### 3.6 Stationarity test (ADF test)
A series is **stationary** when its mean and variance do not change over time. Many statistical models (like ARIMA) need it.
- p-value < 0.05 -> stationary
- p-value >= 0.05 -> not stationary -> we use the **difference** (today - yesterday)

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    print(f"{name}: ADF statistic = {result[0]:.3f}, p-value = {result[1]:.4f}",
          "-> stationary" if result[1] < 0.05 else "-> NOT stationary")

adf_test(data["close"], "Price")
adf_test(data["close"].diff(), "Price difference (1 day)")

### 3.7 ACF and PACF (of the daily difference)
These charts show how much today's value is related to the previous days (lags). They help to choose the ARIMA parameters (p, q).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(data["close"].diff().dropna(), lags=30, ax=axes[0])
plot_pacf(data["close"].diff().dropna(), lags=30, ax=axes[1])
plt.show()

## 4. Feature creation (for the machine learning models)
Machine learning models need columns (features). For a time series we create them from the **past**:
- **lag features** - the price 1, 2, 3, 5, 10 and 21 business days ago
- **rolling features** - average and standard deviation of the last 7 / 30 days (using `shift(1)` so we only use past days)
- **calendar features** - day of the week and month

In [ ]:
features = pd.DataFrame(index=data.index)
features["close"] = data["close"]

for lag in [1, 2, 3, 5, 10, 21]:
    features[f"lag_{lag}"] = data["close"].shift(lag)

features["rolling_mean_7"] = data["close"].shift(1).rolling(7).mean()
features["rolling_mean_30"] = data["close"].shift(1).rolling(30).mean()
features["rolling_std_7"] = data["close"].shift(1).rolling(7).std()
features["day_of_week"] = features.index.dayofweek
features["month"] = features.index.month

features = features.dropna()   # the first 30 rows have no history
print("Shape:", features.shape)
features.head()

## 5. Data splitting based on time (80% / 15% / 5%)
We **never shuffle** a time series - the model must learn from the past and be tested on the future.
- **Train** - the oldest 80%
- **Test** - the next 15%
- **Unseen** - the newest 5% (kept for the final check)

In [ ]:
n = len(features)
train_end = int(n * 0.80)
test_end = int(n * 0.95)

train = features.iloc[:train_end]
test = features.iloc[train_end:test_end]
unseen = features.iloc[test_end:]

print(f"Train : {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} days)")
print(f"Test  : {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} days)")
print(f"Unseen: {unseen.index.min().date()} -> {unseen.index.max().date()}  ({len(unseen)} days)")

plt.figure(figsize=(14, 5))
plt.plot(train["close"], label="Train (80%)")
plt.plot(test["close"], label="Test (15%)")
plt.plot(unseen["close"], label="Unseen (5%)")
plt.title("Time based split")
plt.legend()
plt.show()

In [ ]:
feature_cols = [col for col in features.columns if col != "close"]

X_train, y_train = train[feature_cols], train["close"]
X_test, y_test = test[feature_cols], test["close"]
X_unseen, y_unseen = unseen[feature_cols], unseen["close"]
print("Features:", feature_cols)

## 6. Evaluation function
- **MAE** - average error in USD
- **RMSE** - like MAE but punishes big errors more
- **MAPE %** - average error in %
- **R2** - how much of the variation is explained (1 = perfect)

In [ ]:
def evaluate_forecast(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE %": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "R2": r2_score(y_true, y_pred),
    }

train_results = {}
test_results = {}
test_predictions = {}

## 7. Modeling

### 7.1 Baseline 1 - Naive forecast
Tomorrow's price = today's price. This simple idea is very hard to beat for prices, so every model must be compared with it.

In [ ]:
train_results["Naive (yesterday)"] = evaluate_forecast(y_train, train["lag_1"])
test_results["Naive (yesterday)"] = evaluate_forecast(y_test, test["lag_1"])
test_predictions["Naive (yesterday)"] = test["lag_1"].values
test_results["Naive (yesterday)"]

### 7.2 Baseline 2 - Moving average (last 7 days)

In [ ]:
train_results["Moving Average (7)"] = evaluate_forecast(y_train, train["rolling_mean_7"])
test_results["Moving Average (7)"] = evaluate_forecast(y_test, test["rolling_mean_7"])
test_predictions["Moving Average (7)"] = test["rolling_mean_7"].values
test_results["Moving Average (7)"]

### 7.3 Holt Exponential Smoothing (multi-step forecast)
Holt's model learns the **level** and the **trend** of the series. Here it forecasts the **whole test period at once** from the end of the training data
(no new information during the test period). This is much harder than predicting one day ahead.

In [ ]:
train_series = train["close"].asfreq("B")

holt = ExponentialSmoothing(train_series, trend="add", damped_trend=True).fit()
holt_train_pred = holt.fittedvalues
holt_test_pred = holt.forecast(len(test))

train_results["Holt (multi-step)"] = evaluate_forecast(y_train, holt_train_pred)
test_results["Holt (multi-step)"] = evaluate_forecast(y_test, holt_test_pred.values)
test_predictions["Holt (multi-step)"] = holt_test_pred.values
test_results["Holt (multi-step)"]

### 7.4 ARIMA
ARIMA(p, d, q):
- **p** - how many past values (AR part)
- **d** - how many times we take the difference (1, because the price is not stationary)
- **q** - how many past errors (MA part)

We try 2 ways:
1. **Multi-step** - forecast the whole test period at once (like Holt).
2. **One-step ahead** - every day we give the model the real price of yesterday and predict today (same parameters, no re-training).

In [ ]:
arima = ARIMA(train_series, order=(1, 1, 1)).fit()
print(arima.summary().tables[1])

# the first fitted value of a differenced model is not meaningful, so we skip it
arima_train_pred = arima.fittedvalues[1:]
train_results["ARIMA(1,1,1) multi-step"] = evaluate_forecast(y_train[1:], arima_train_pred)
train_results["ARIMA(1,1,1) one-step"] = train_results["ARIMA(1,1,1) multi-step"]

# 1) multi-step forecast
arima_multi = arima.forecast(steps=len(test))
test_results["ARIMA(1,1,1) multi-step"] = evaluate_forecast(y_test, arima_multi.values)
test_predictions["ARIMA(1,1,1) multi-step"] = arima_multi.values

# 2) one-step ahead: add the test data to the model (same parameters) and predict each day
test_series = test["close"].asfreq("B")
arima_with_test = arima.append(test_series)
arima_one_step = arima_with_test.predict(start=test_series.index[0], end=test_series.index[-1])
test_results["ARIMA(1,1,1) one-step"] = evaluate_forecast(y_test, arima_one_step.values)
test_predictions["ARIMA(1,1,1) one-step"] = arima_one_step.values

pd.DataFrame(test_results).T.round(3)

### 7.5 Machine learning models (with lag features)
We use 6 regression models. They predict today's price from the features of section 4 (one-step ahead).

**Cross validation for time series:** `TimeSeriesSplit` always trains on the past and validates on the next block (it never uses the future).

In [ ]:
ml_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0, max_iter=10000),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=RANDOM_STATE),
}

tscv = TimeSeriesSplit(n_splits=5)
for name, model in ml_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring="neg_root_mean_squared_error")
    print(f"{name:20s} CV RMSE = {-scores.mean():.2f} (+/- {scores.std():.2f})")

### 7.6 Hyperparameter tuning (Grid Search with TimeSeriesSplit)

In [ ]:
param_grids = {
    "Ridge": {"alpha": [0.1, 1, 10, 100]},
    "Lasso": {"alpha": [0.1, 1, 10]},
    "Random Forest": {"n_estimators": [100, 200], "max_depth": [5, 10, None]},
    "Gradient Boosting": {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]},
    "XGBoost": {"n_estimators": [200, 400], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]},
}

for name, grid_values in param_grids.items():
    grid = GridSearchCV(ml_models[name], grid_values, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    grid.fit(X_train, y_train)
    ml_models[name] = grid.best_estimator_
    print(f"{name:20s} best params = {grid.best_params_}")

### 7.7 Train the machine learning models - training and testing evaluation

In [ ]:
for name, model in ml_models.items():
    model.fit(X_train, y_train)
    train_results[name] = evaluate_forecast(y_train, model.predict(X_train))
    test_pred = model.predict(X_test)
    test_results[name] = evaluate_forecast(y_test, test_pred)
    test_predictions[name] = test_pred
    print("Trained:", name)

## 8. Compare all the models

**Important lesson:** tree models (Random Forest, Gradient Boosting, XGBoost) **cannot predict values higher than what they saw in training**.
When the gold price in the test period is higher than ever before, they stay "stuck" at the old maximum - you will see a big error for them.
Linear models and the naive forecast do not have this problem.

### Training evaluation

In [ ]:
pd.DataFrame(train_results).T.sort_values("RMSE").round(3)

### Testing evaluation

In [ ]:
comparison = pd.DataFrame(test_results).T.sort_values("RMSE")
comparison.round(3)

In [ ]:
comparison["RMSE"].sort_values(ascending=False).plot(kind="barh", figsize=(10, 6), color="goldenrod")
plt.title("Test RMSE (lower is better)")
plt.xlabel("RMSE (USD)")
plt.show()

In [ ]:
# Actual vs predictions on the test period
plt.figure(figsize=(14, 6))
plt.plot(test.index, y_test, label="Actual", color="black", linewidth=2)
for name in ["Naive (yesterday)", "ARIMA(1,1,1) multi-step", "Holt (multi-step)", "Linear Regression", "Random Forest", "XGBoost"]:
    plt.plot(test.index, test_predictions[name], label=name, alpha=0.8)
plt.title("Test period - actual vs predicted")
plt.legend()
plt.show()

## 9. Save the best machine learning model
We choose the **machine learning model with the lowest test RMSE** and save it with `joblib`.
(The naive forecast is not a trained model, but we keep it as the baseline to compare with.)

In [ ]:
ml_comparison = comparison.loc[list(ml_models.keys())].sort_values("RMSE")
best_model_name = ml_comparison.index[0]
best_model = ml_models[best_model_name]
print("Best machine learning model:", best_model_name)

os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/best_model.pkl")
joblib.dump(feature_cols, "models/feature_cols.pkl")
print("Saved to models/best_model.pkl")

## 10. Load the model and predict the unseen 5% (newest data)

In [ ]:
loaded_model = joblib.load("models/best_model.pkl")
loaded_features = joblib.load("models/feature_cols.pkl")

unseen_pred = loaded_model.predict(X_unseen[loaded_features])

pd.DataFrame({
    f"{best_model_name} - test": test_results[best_model_name],
    f"{best_model_name} - unseen": evaluate_forecast(y_unseen, unseen_pred),
    "Naive - unseen": evaluate_forecast(y_unseen, unseen["lag_1"]),
}).round(3)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(unseen.index, y_unseen, label="Actual", color="black", linewidth=2)
plt.plot(unseen.index, unseen_pred, label=f"Predicted ({best_model_name})", color="red")
plt.title("Unseen 5% - actual vs predicted")
plt.legend()
plt.show()

result = pd.DataFrame({"Actual": y_unseen, "Predicted": unseen_pred.round(2)})
result["Error"] = (result["Actual"] - result["Predicted"]).round(2)
result.tail(10)

## 11. Forecast the next 30 business days (ARIMA on all the data)
For the real future we do not have the real prices, so we use ARIMA trained on **all** the data and show the forecast with a 95% confidence interval.
The interval gets wider the further we look - the future is uncertain.

In [ ]:
full_arima = ARIMA(data["close"], order=(1, 1, 1)).fit()
future = full_arima.get_forecast(steps=30).summary_frame()

plt.figure(figsize=(14, 5))
plt.plot(data["close"].iloc[-250:], label="Last 250 days")
plt.plot(future["mean"], label="Forecast", color="red")
plt.fill_between(future.index, future["mean_ci_lower"], future["mean_ci_upper"], color="red", alpha=0.2, label="95% interval")
plt.title("Gold price - next 30 business days")
plt.legend()
plt.show()

future[["mean", "mean_ci_lower", "mean_ci_upper"]].round(2).head(10)

# Summary
1. We cleaned the data (sorted dates, removed duplicates, filled holidays with the previous price).
2. EDA showed a strong **upward trend**, no clear seasonality, and the price is **not stationary** (its difference is).
3. We split the data **by time** (80 / 15 / 5), never shuffling.
4. We compared baselines (Naive, Moving Average), statistical models (Holt, ARIMA) and 6 machine learning models.
5. Predicting **one day ahead** is much easier than predicting many days ahead. The naive forecast is a very strong baseline for prices.
6. Tree models cannot go above the highest price they saw in training.

**Ideas to try next:** predict the **daily return (%)** instead of the price (then trees work better), add other features (USD index, oil price, interest rates), try SARIMA or Prophet.